In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
df = pd.read_csv("../../Datasets_For_Model_Training/Final_Merged_Dataset.csv")
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature
0,01-04-2015,8361.0,70.07,130.566857,8112.0,166.53,28.57
1,01-05-2015,8381.0,77.22,160.792286,8165.0,155.03,27.95
2,01-06-2015,8302.0,77.55,98.240286,8257.0,160.31,27.31
3,01-07-2015,8953.0,73.18,37.804286,8901.0,166.41,27.68
4,01-08-2015,8535.0,72.58,63.944286,8531.0,167.12,27.85


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     129 non-null    object 
 1   Electricity_Requirement  129 non-null    float64
 2   Humidity                 129 non-null    float64
 3   Rainfall                 129 non-null    float64
 4   Electricity_Supply       129 non-null    float64
 5   Solar_Irradiance         129 non-null    float64
 6   Temperature              129 non-null    float64
dtypes: float64(6), object(1)
memory usage: 7.2+ KB


In [6]:
# Explicit date conversion preserving monthly dates
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

In [7]:
print("Min date:", df["Date"].min())
print("Max date:", df["Date"].max())
print("\nFirst 5 rows:")
print(df["Date"].head())
print("\nLast 5 rows:")
print(df["Date"].tail())

Min date: 2015-04-01 00:00:00
Max date: 2025-12-01 00:00:00

First 5 rows:
0   2015-04-01
1   2015-05-01
2   2015-06-01
3   2015-07-01
4   2015-08-01
Name: Date, dtype: datetime64[ns]

Last 5 rows:
124   2025-08-01
125   2025-09-01
126   2025-10-01
127   2025-11-01
128   2025-12-01
Name: Date, dtype: datetime64[ns]


In [8]:
# Extract Year and Month
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

In [9]:
# Cyclical Encoding of Month
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

print(df[["Month", "Month_sin", "Month_cos"]].drop_duplicates().sort_values("Month"))

    Month     Month_sin     Month_cos
9       1  5.000000e-01  8.660254e-01
10      2  8.660254e-01  5.000000e-01
11      3  1.000000e+00  6.123234e-17
0       4  8.660254e-01 -5.000000e-01
1       5  5.000000e-01 -8.660254e-01
2       6  1.224647e-16 -1.000000e+00
3       7 -5.000000e-01 -8.660254e-01
4       8 -8.660254e-01 -5.000000e-01
5       9 -1.000000e+00 -1.836970e-16
6      10 -8.660254e-01  5.000000e-01
7      11 -5.000000e-01  8.660254e-01
8      12 -2.449294e-16  1.000000e+00


In [11]:
# Festival Feature Creation
deepavali_months = {
    2015: 11, 2016: 10, 2017: 10, 2018: 11, 2019: 10,
    2020: 11, 2021: 11, 2022: 10, 2023: 11, 2024: 10, 2025: 10
}

df["Festival"] = 0
df.loc[df["Month"] == 1, "Festival"] = 1

for year, festival_month in deepavali_months.items():
    df.loc[(df["Year"] == year) & (df["Month"] == festival_month), "Festival"] = 1

print("Festival Count:")
print(df["Festival"].value_counts())
print("\nFirst 20 rows of (Date, Month, Festival):")
print(df[["Date", "Month", "Festival"]].head(20))

Festival Count:
Festival
0    108
1     21
Name: count, dtype: int64

First 20 rows of (Date, Month, Festival):
         Date  Month  Festival
0  2015-04-01      4         0
1  2015-05-01      5         0
2  2015-06-01      6         0
3  2015-07-01      7         0
4  2015-08-01      8         0
5  2015-09-01      9         0
6  2015-10-01     10         0
7  2015-11-01     11         1
8  2015-12-01     12         0
9  2016-01-01      1         1
10 2016-02-01      2         0
11 2016-03-01      3         0
12 2016-04-01      4         0
13 2016-05-01      5         0
14 2016-06-01      6         0
15 2016-07-01      7         0
16 2016-08-01      8         0
17 2016-09-01      9         0
18 2016-10-01     10         1
19 2016-11-01     11         0


In [12]:
# Create Historical Demand Lags (Autoregressive Predictors)
df["Demand_Lag_1"] = df["Electricity_Requirement"].shift(1)
df["Demand_Lag_2"] = df["Electricity_Requirement"].shift(2)
df["Demand_Lag_3"] = df["Electricity_Requirement"].shift(3)

# Create Historical Demand Rolling Means (Shifted by 1 to prevent leakage)
df["Demand_Rolling_3"] = df["Electricity_Requirement"].shift(1).rolling(3).mean()
df["Demand_Rolling_6"] = df["Electricity_Requirement"].shift(1).rolling(6).mean()
df["Demand_Rolling_12"] = df["Electricity_Requirement"].shift(1).rolling(12).mean()

In [13]:
# Remove rows where lag/rolling features are unavailable (Single dropna step)
df = df.dropna().reset_index(drop=True)

print("DataFrame shape after single dropna():", df.shape)
print("Null counts:\n", df.isnull().sum())

DataFrame shape after single dropna(): (117, 18)
Null counts:
 Date                       0
Electricity_Requirement    0
Humidity                   0
Rainfall                   0
Electricity_Supply         0
Solar_Irradiance           0
Temperature                0
Year                       0
Month                      0
Month_sin                  0
Month_cos                  0
Festival                   0
Demand_Lag_1               0
Demand_Lag_2               0
Demand_Lag_3               0
Demand_Rolling_3           0
Demand_Rolling_6           0
Demand_Rolling_12          0
dtype: int64


In [15]:
df.to_csv("../Demand_Forecasting_Validation/Demand.csv",index=False)

In [16]:
df.head()

,Date,Electricity_Requirement,Humidity,Rainfall,Electricity_Supply,Solar_Irradiance,Temperature,Year,Month,Month_sin,Month_cos,Festival,Demand_Lag_1,Demand_Lag_2,Demand_Lag_3,Demand_Rolling_3,Demand_Rolling_6,Demand_Rolling_12
0,2016-04-01,9847.4,61.62,22.442000,9842.3,191.19,31.11,2016,4,8.660254e-01,-0.500000,0,9688.0,8386.0,8151.0,8741.666667,8041.166667,8273.083333
1,2016-05-01,9292.9,70.07,136.451714,9294.1,166.19,29.45,2016,5,5.000000e-01,-0.866025,0,9847.4,9688.0,8386.0,9307.133333,8294.066667,8396.950000
2,2016-06-01,8744.5,74.05,83.717714,8742.8,147.03,27.64,2016,6,1.224647e-16,-1.000000,0,9292.9,9847.4,9688.0,9609.433333,8757.716667,8472.941667
3,2016-07-01,9006.6,73.96,100.907714,9006.6,149.69,27.29,2016,7,-5.000000e-01,-0.866025,0,8744.5,9292.9,9847.4,9294.933333,9018.300000,8509.816667
4,2016-08-01,9207.2,72.70,55.885143,9207.2,177.66,27.55,2016,8,-8.660254e-01,-0.500000,0,9006.6,8744.5,9292.9,9014.666667,9160.900000,8514.283333


In [12]:
df.tail()

In [13]:
# Target Selection (Demand Target Only)
y = df["Electricity_Requirement"]

# Feature Selection (Excluding Electricity_Supply and Current Demand)

In [15]:
feature_cols = [
    "Humidity",
    "Rainfall",
    "Solar_Irradiance",
    "Temperature",
    "Year",
    "Month_sin",
    "Month_cos",
    "Festival",
    "Demand_Lag_1",
    "Demand_Lag_2",
    "Demand_Lag_3",
    "Demand_Rolling_3",
    "Demand_Rolling_6",
    "Demand_Rolling_12"
]

X = df[feature_cols]

In [16]:
print("Final features:")
print(X.columns.tolist())
print("\nX shape:", X.shape)
print("y shape:", y.shape)

Final features:
['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Festival', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3', 'Demand_Rolling_3', 'Demand_Rolling_6', 'Demand_Rolling_12']

X shape: (117, 14)
y shape: (117,)


# Chronological 24-Month Train/Test Split

In [18]:
test_size = 24
split = len(df) - test_size

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print(
    "Training period:",
    df["Date"].iloc[:split].min().strftime("%Y-%m-%d"),
    "to",
    df["Date"].iloc[:split].max().strftime("%Y-%m-%d")
)

print(
    "Testing period:",
    df["Date"].iloc[split:].min().strftime("%Y-%m-%d"),
    "to",
    df["Date"].iloc[split:].max().strftime("%Y-%m-%d")
)

Training data: (93, 14)
Testing data : (24, 14)
Training period: 2016-04-01 to 2023-12-01
Testing period: 2024-01-01 to 2025-12-01


In [19]:
# Save preprocessed datasets
joblib.dump(X_train, "Demand_X_train.pkl")
joblib.dump(X_test, "Demand_X_test.pkl")

joblib.dump(y_train, "Demand_y_train.pkl")
joblib.dump(y_test, "Demand_y_test.pkl")

print("Preprocessed demand datasets saved successfully.")

Preprocessed demand datasets saved successfully.


In [20]:
# Final Preprocessing Verification Output
supply_leakage = "Electricity_Supply" in X.columns
current_demand_leakage = "Electricity_Requirement" in X.columns
chronological_pass = (X_train.shape[0] == 93) and (X_test.shape[0] == 24)

print("========================================")
print("DEMAND PREPROCESSING VERIFICATION")
print("========================================")
print(f"DataFrame shape: {df.shape}")
print(f"Start date: {df['Date'].min().strftime('%Y-%m-%d')}")
print(f"End date:   {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures:\n{X.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum().to_dict()}")
print(f"\nTraining shape: {X_train.shape}")
print(f"Testing shape : {X_test.shape}")
print(f"\nTraining period: {df['Date'].iloc[:split].min().strftime('%Y-%m-%d')} to {df['Date'].iloc[:split].max().strftime('%Y-%m-%d')}")
print(f"Testing period : {df['Date'].iloc[split:].min().strftime('%Y-%m-%d')} to {df['Date'].iloc[split:].max().strftime('%Y-%m-%d')}")
print(f"\nSupply leakage: {'FAIL' if supply_leakage else 'PASS'}")
print(f"Current demand leakage: {'FAIL' if current_demand_leakage else 'PASS'}")
print(f"Chronological split: {'PASS' if chronological_pass else 'FAIL'}")
print("\nSaved files:")
print("Demand_X_train.pkl")
print("Demand_X_test.pkl")
print("Demand_y_train.pkl")
print("Demand_y_test.pkl")
print("========================================")

DEMAND PREPROCESSING VERIFICATION
DataFrame shape: (117, 18)
Start date: 2016-04-01
End date:   2025-12-01

X shape: (117, 14)
y shape: (117,)

Features:
['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Festival', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3', 'Demand_Rolling_3', 'Demand_Rolling_6', 'Demand_Rolling_12']

Missing values:
{'Date': 0, 'Electricity_Requirement': 0, 'Humidity': 0, 'Rainfall': 0, 'Electricity_Supply': 0, 'Solar_Irradiance': 0, 'Temperature': 0, 'Year': 0, 'Month': 0, 'Month_sin': 0, 'Month_cos': 0, 'Festival': 0, 'Demand_Lag_1': 0, 'Demand_Lag_2': 0, 'Demand_Lag_3': 0, 'Demand_Rolling_3': 0, 'Demand_Rolling_6': 0, 'Demand_Rolling_12': 0}

Training shape: (93, 14)
Testing shape : (24, 14)

Training period: 2016-04-01 to 2023-12-01
Testing period : 2024-01-01 to 2025-12-01

Supply leakage: PASS
Current demand leakage: PASS
Chronological split: PASS

Saved files:
Demand_X_train.pkl
Demand_X_test.pkl
Demand_y_train.

In [1]:
import os

files_to_check = [
    "Demand_LSTM_Final.keras",
    "Demand_LSTM_X_Scaler.pkl",
    "Demand_LSTM_y_Scaler.pkl",
    "Demand_LSTM_Hyperparameters.json"
]

print("========================================")
print("FILE CHECK")
print("========================================")

for file in files_to_check:
    print(
        f"{'PASS' if os.path.exists(file) else 'MISS'} : {file}"
    )

FILE CHECK
PASS : Demand_LSTM_Final.keras
PASS : Demand_LSTM_X_Scaler.pkl
PASS : Demand_LSTM_y_Scaler.pkl
PASS : Demand_LSTM_Hyperparameters.json
